# Importance Analysis

# Setup

## Imports

In [ ]:
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from thesis_project import (
    config as global_config,
)

## Paths

In [ ]:
TICKER = "fbtp"
# NOTE: Use "notebook-name" if needed
NOTEBOOK = "importance-mi"

# Lobs directories
LOB_DIR = global_config.RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "futures"
LOB_CTD_DIR = LOB_DIR / "ctd"

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = global_config.FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = global_config.TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    # NOTE: Create a purpose specific directory if needed
    # OUT_INT_DIR = config.INT_DIR / TICKER / NOTEBOOK
    # OUT_INT_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_PRO_DIR = config.PRO_DIR / TICKER / NOTEBOOK
    # OUT_PRO_DIR.mkdir(parents=True, exist_ok=True)

## Settings

In [ ]:
# Matplotlib global settings
# config.MPL contains a personalized default configuration for matplotlib
plt.rcdefaults()
plt.style.use("seaborn-v0_8-paper")

In [ ]:
# Warnings settings
warnings.simplefilter("always", UserWarning)

# Main

## Importance Mutual Information

In [ ]:
import datetime


def float_to_time(x):
    return str(datetime.timedelta(seconds=int(x)))


print(float_to_time(65532.52))

In [ ]:
from natsort import natsorted

TICKER = "fbtp"

# run 1 and 2 are preliminary
runs = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

partial = {}
for run in runs:
    partial[run] = pd.read_parquet(
        global_config.INT_DIR
        / TICKER
        / "importance-mi"
        / "runs"
        / f"run_{run:03d}"
        / "results.parquet"
    )


results = pd.concat(partial.values(), ignore_index=True)

pairs = results[["horizon", "target"]].drop_duplicates().values
pairs = natsorted(pairs, key=lambda x: (x[1], x[0]))
n = 20
key = "cmi"
print(f"Top {n} features by {key} for each horizon and target:\n")
for h, t in pairs:
    mask = (results["horizon"] == h) & (results["target"] == t)
    data = results[mask].sort_values([key], ascending=False)
    data = data.drop(columns=["horizon", "target", "n_samples", "timestamp"])
    data = data.reset_index(drop=True)
    data.index += 1
    data.index.name = "rank"

    title = f"Top {n} features by {key} for horizon {h}s and target {t}"
    # data = data.head(n)
    mask = data["feature"].str.contains("aof_lvl")
    data = data[mask]
    display(data.style.set_caption(title))
    # data.to_csv(OUT_TAB_DIR / f"{t}_top_{n}_features_vs_by_{key}_horizon_{h}s.csv")

## Importance XGBoost

In [ ]:
import yaml
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)

from thesis_project.importance_xgb import config, data_paths

In [ ]:
run = "prova"
base_path = global_config.INT_DIR / TICKER / "importance-xgb" / "runs"
# path = base_path / f"run_{run:03d}"
path = base_path / f"run_{run}"

with open(path / "config.yaml") as f:
    config_dict = yaml.safe_load(f)
exp_config_obj = config.AnalysisConfig(**config_dict)

with open(path / "data_paths.yaml") as f:
    data_paths_dict = yaml.safe_load(f)
exp_data_paths = data_paths.DataPathsConfig(
    TICKER, data_paths_dict["target"], data_paths_dict["features"]
)

with open(path / "metadata.yaml") as f:
    exp_metadata = yaml.safe_load(f)

cls_labels = ["Low", "Medium", "High"]

### Metrics

In [ ]:
y_true = pd.read_parquet(path / "y_cls_test.parquet").iloc[:, 0]
y_proba = pd.read_parquet(path / "y_cls_proba_test.parquet")
y_pred = np.argmax(y_proba, axis=1)

metrics = {
    "logloss": log_loss(y_true, y_proba, labels=range(exp_config_obj.n_quantile)),
    "accuracy": accuracy_score(y_true, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, average="macro"),
    "recall": recall_score(y_true, y_pred, average="macro"),
    "f1": f1_score(y_true, y_pred, average="macro"),
    "auc": roc_auc_score(y_true, y_proba, average="macro", multi_class="ovo"),
    "mcc": matthews_corrcoef(y_true, y_pred),
}

df = pd.DataFrame.from_dict(metrics, orient="index", columns=[""])
df.index.name = "metric"
display(df)

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred, normalize="true")

fig, ax = plt.subplots(figsize=(6, 6))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cls_labels)
disp.plot(ax=ax, values_format=".2f", cmap="Blues")

ax.set_title("Confusion Matrix", fontsize=16)
ax.set_xlabel("Predicted label", fontsize=12)
ax.set_ylabel("True label", fontsize=12)

plt.tight_layout()
plt.show()

### Model importance

In [ ]:
importance_xgb = pd.read_csv(path / "importance_xgb.csv")

n = 25
metric = "gain"
data = importance_xgb.sort_values(by=metric, ascending=False).reset_index(drop=True).head(n)
data.index += 1
data.index.name = "rank"
# display(data)

fig, ax = plt.subplots(figsize=(12, 8))

ax.barh(data["feature"], data[metric])

ax.invert_yaxis()
ax.set_xlabel(metric.title())
ax.set_ylabel("Feature")
ax.set_title(f"Top {n} features by {metric}")

fig.tight_layout()
plt.show()

### Shap Importance

In [ ]:
import shap

n = 25

explainer = joblib.load(path / "importance_shap.pkl")
for cls in range(exp_config_obj.n_quantile):
    shap.plots.beeswarm(explainer[:, :, cls], max_display=n)

In [ ]:
print(exp_config_obj.feature_transforms)
print(explainer.feature_names)

In [ ]:
def keys_transform(config_obj: config.AnalysisConfig):
    keys = []
    for transform in config_obj.feature_transforms:
        if transform["name"] == "delta":
            for delta in transform["params"]["deltas"]:
                keys.append(f"delta_{delta}s")

        if transform["name"] == "rolling":
            for stat in transform["params"]["stats"]:
                for window in transform["params"]["windows"]:
                    keys.append(f"roll_{stat}_{window}s")

        if transform["name"] == "ratio":
            for ref in transform["params"]["references"]:
                keys.append(f"{ref}_ratio")

    return keys

In [ ]:
cls_dict = {i: f"Quantile {i+1}" for i in range(exp_config_obj.n_quantile)}

all_features = explainer.feature_names
# keys = list(exp_data_paths.features.keys())
keys = keys_transform(exp_config_obj)

groups = {key: [i for i, feature in enumerate(all_features) if key in feature] for key in keys}

data = np.column_stack([explainer.data[:, idxs].sum(axis=1) for idxs in groups.values()])

n = 11
for cls in range(exp_config_obj.n_quantile):
    values = np.column_stack(
        [explainer.values[:, idxs, cls].sum(axis=1) for idxs in groups.values()]
    )

    grouped = shap.Explanation(values=values, data=None, feature_names=keys)

    size = 14
    shap.plots.beeswarm(grouped, max_display=n, show=False)
    plt.title(f"Grouped SHAP Beeswarm - {cls_dict[cls]}", size=size)
    plt.tight_layout()
    plt.show()

    shap.plots.bar(grouped, max_display=n, show=False)
    plt.title(f"Grouped SHAP Barplot - {cls_dict[cls]}", size=size)
    plt.tight_layout()
    plt.show()